# R3 · 3 · Avaliação e gráficos da rodada 3

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 e 2):
1. suba **`r3_lib.py`** (pasta `data_imdc_2026/RODADA3/`) e confira que o `r2_lib.py` mais recente está lá;
2. **crie à mão** as pastas `resultados_uf_r3`, `revisoes_r3`, `cache_r3` e `avaliacao_r3`.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r3/uf/` (o CompUF4 e o Comp5 também as enxergam). As revisões de julho a outubro vão para `revisoes_r3/uf/`, fora da régua oficial. As tabelas vão para `avaliacao_r3/`, com data e hora no nome e uma cópia `_ultimo`.

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (variáveis, um arquivo por método × corte do detector, parâmetros em json, um arquivo por modelo × temporada, um PNG por UF) e o que já existe é pulado.

**Dados externos (opcional):** para testar sinais que não estão no FTP (previsão de ENSO do IRI, sorotipo, vacinação, positividade de exames), ponha em `cache_r3/externos_r3.csv` colunas `uf_code`, `season` e, se quiser, `corte` (`jun`/`jul`/`ago`/`set`/`out`), mais uma coluna por sinal. Cada valor deve ser o que se sabia **naquele corte**. Apague `cache_r3/detector_*` e `cache_r3/sinais_detector.json` para refazer.

CPU comum, ~10–20 min. **Rode depois do R3_2.**

- **Régua oficial (teste 2022–2025, corte de junho):** R × BB, por fase (pico, intermediária, entressafra), erro no pico, erro no total da temporada e cobertura de cada quantil, comparando as misturas com o normal (ENS_grafos3_λ) e o BB.
- **Escolha honesta:** a variante recomendada é a de menor R na **afinação**, e não a que for melhor no teste.
- **Revisões:** quanto cada mês a mais de dados (jul → out) melhora a previsão da temporada.
- **Gráficos:** uma figura por UF e uma do Brasil (2022–2025), com a recomendada, o normal e o BB, e o que o detector disse em junho (probabilidade de temporada grande e tamanho previsto). Saída: `avaliacao_r3/graficos_r3_<data>/` (PNGs, PDF e totais por temporada).


In [ ]:
NOTEBOOK = 'r3_avaliacao'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r3', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
sys.path.insert(0, PASTA_LOCAL)
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_REV = os.path.join(PASTA_DRIVE, 'revisoes_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV3 = os.path.join(PASTA_DRIVE, 'avaliacao_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R3, 'uf'), os.path.join(PASTA_REV, 'uf'), PASTA_C3, PASTA_AV3):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3
for m in (V, L, CM, U, S, R2, R3):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
def detector_escolhido():
    esc = R3.json_le(os.path.join(PASTA_C3, 'detector_escolhido.json'))
    assert esc, 'Rode antes o R3_1_Detector.'
    P = pd.concat([pd.read_parquet(os.path.join(PASTA_C3, f'detector_{m}_{k}.parquet')) for k, m in esc.items()], ignore_index=True)
    return esc, P


In [ ]:
# @title 2 · Ler normal, BB, climatologia, misturas e revisões
NORMAL = R3.normal_ens_grafos3(ctx, PASTA_DRIVE, PASTA_AV2, PASTA_COMP, PASTA_EXT, PASTA_C3, TUNING + TESTE)
MIX = pd.concat([pd.read_parquet(f) for p in (PASTA_R3, PASTA_REV) for f in glob.glob(os.path.join(p, 'uf', '*.parquet'))],
                ignore_index=True)
MIX['date'] = pd.to_datetime(MIX.date)
TODOS = pd.concat([NORMAL, MIX[L.COLS_PREV]], ignore_index=True)
PAR = R3.json_le(os.path.join(PASTA_C3, 'mistura_parametros.json'))
RECOMENDADA = R3.nome_mistura(min(PAR, key=lambda v: min(r for _, r in PAR[v]['grade'])), 'jun')
ESC, DET = detector_escolhido()
print('modelos:', sorted(TODOS.modelo.unique()), '\nrecomendada (escolhida na afinação):', RECOMENDADA)


In [ ]:
# @title 3 · Régua oficial no teste (corte de junho): geral, fases, pico, total e cobertura
OF = TODOS[TODOS.season.isin(TESTE) & ~TODOS.modelo.str.contains('@')]
_, RES = V.avaliar(OF[L.COLS_PREV], ctx.ver_uf, ref=('BB', 'climatologia'))
RES = RES[RES.temporadas == len(TESTE)]
TT = S.com_verdade(OF, D)
pont = pd.DataFrame([dict(modelo=m, **S.metricas(g)) for m, g in TT.groupby('modelo')])
LIM = R2.limiares(D)
FASE = R2.metricas_fase(R2.com_fase(TT, D, LIM), ['modelo'])
PICO = R2.erros_pico(TT, 'season')
COB = R2.cobertura_quantis(TT)
f1 = FASE.pivot_table(index='modelo', columns='fase', values=['WIS_log', 'vies_pct', 'IC90'])
f1.columns = [f'{a}_{b}' for a, b in f1.columns]
TAB = RES[['modelo', 'R_vs_BB', 'UFs_melhor_que_BB', 'R_vs_climatologia']].merge(pont, on='modelo') \
    .merge(f1.reset_index(), on='modelo', how='left').merge(PICO, on='modelo', how='left').sort_values('R_vs_BB')
for nome, t in (('temporada_teste_r3', TAB), ('temporada_teste_por_fase_r3', FASE), ('temporada_teste_pico_r3', PICO),
                ('temporada_teste_cobertura_r3', COB)):
    R3.salvar_tabela(t, PASTA_AV3, nome)
cols = ['modelo', 'R_vs_BB', 'UFs_melhor_que_BB', 'WAPE', 'IC90', 'WIS_log_alta (pico)', 'WIS_log_entressafra',
        'IC90_alta (pico)', 'erro_total_mediano_pct', 'pico_no_IC90_pct']
TAB[[c for c in cols if c in TAB]].round(3)


In [ ]:
# @title 4 · Revisões mensais: R × BB e erro no total da temporada por corte
lin = []
for v in PAR:
    for corte in R3.CORTES:
        nome = R3.nome_mistura(v, corte)
        g = TODOS[(TODOS.modelo == nome) & TODOS.season.isin(TESTE)]
        if g.empty:
            continue
        _, r = V.avaliar(pd.concat([g.assign(modelo='x'), NORMAL[NORMAL.modelo == 'BB']])[L.COLS_PREV], ctx.ver_uf, ref=('BB',))
        pk = R2.erros_pico(S.com_verdade(g.assign(modelo='x'), D), 'season')
        lin.append(dict(variante=v, corte=corte, R_vs_BB=float(r.set_index('modelo').loc['x', 'R_vs_BB']),
                        erro_total_mediano_pct=float(pk.erro_total_mediano_pct.iloc[0]), pico_no_IC90_pct=float(pk.pico_no_IC90_pct.iloc[0])))
REV = pd.DataFrame(lin)
R3.salvar_tabela(REV, PASTA_AV3, 'revisoes_mensais')
REV.pivot_table(index='variante', columns='corte', values=['R_vs_BB', 'erro_total_mediano_pct'])[
    [(a, k) for a in ('R_vs_BB', 'erro_total_mediano_pct') for k in R3.CORTES]].round(3)


In [ ]:
# @title 5 · Gráficos (26 UFs + Brasil), PDF e totais por temporada (pula os PNGs prontos)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import Image, display
PASTA_G = os.path.join(PASTA_AV3, f'graficos_r3_{R3.carimbo()[:8]}')
os.makedirs(PASTA_G, exist_ok=True)
OBS = pd.DataFrame(D.Y[:D.ultimo_obs + 1], index=pd.to_datetime(D.datas[:D.ultimo_obs + 1]), columns=D.unidades)
OBS[0] = OBS.sum(1)
PREV = {f'recomendada · {RECOMENDADA}': TODOS[TODOS.modelo == RECOMENDADA],
        'normal · ENS_grafos3_λ': TODOS[TODOS.modelo == 'ENS_grafos3_λ'],
        'modelo-base oficial · BB': TODOS[TODOS.modelo == 'BB']}
DJ = DET[(DET.corte == 'jun')]
ORDEM = [0] + sorted(D.unidades, key=lambda u: R2.SIGLAS.get(u, str(u)))
tot = []
for u in ORDEM:
    arq = os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png')
    if not os.path.exists(arq) or REFAZER:
        det = DJ[DJ.uf_code == u].set_index('season') if u != 0 else None
        fig = R3.grafico_temporada_uf(OBS[u], PREV, u, arq, TESTE, det=det, titulo=' · Brasil = soma das UFs' if u == 0 else '')
        plt.close(fig)
    tot.append(R2.totais_temporada(OBS[u], PREV, u, TESTE))
with PdfPages(os.path.join(PASTA_G, 'graficos_UFs_r3.pdf.tmp')) as pdf:
    for u in ORDEM:
        img = plt.imread(os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png'))
        f = plt.figure(figsize=(13, 4.6)); plt.imshow(img); plt.axis('off'); pdf.savefig(f, bbox_inches='tight'); plt.close(f)
os.replace(os.path.join(PASTA_G, 'graficos_UFs_r3.pdf.tmp'), os.path.join(PASTA_G, 'graficos_UFs_r3.pdf'))
TOT = pd.concat(tot, ignore_index=True)
TOT.to_csv(os.path.join(PASTA_G, 'totais_temporada_r3.csv'), index=False)
print('Pasta:', PASTA_G)
for u in (0, 35, 31, 41, 42, 23):
    display(Image(os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png')))
TOT.round(0)
